In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Dataset

from torch.optim import Adam

from PIL import Image

import matplotlib.pyplot as plt

import os
import torchvision.transforms as T

from torch.cuda.amp import GradScaler, autocast


In [ ]:
class MyDataset(Dataset):
    def __init__(self, limit=40000):
        self.data = []
        self.root_dir =  "/kaggle/input/datasets/greatgamedota/ffhq-face-data-set/thumbnails128x128"


        all_images = sorted(os.listdir(self.root_dir))

        count = 0
        for img_name in all_images:
            if count >= limit:
                break

            img_path = os.path.join(self.root_dir, img_name)
            self.data.append(img_path)
            count += 1

        self.transform = T.Compose([
            T.Resize((128, 128)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)  # [-1, 1]
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        photo = Image.open(self.data[idx]).convert("RGB")
        photo = self.transform(photo)
        return photo

In [ ]:
dataset = MyDataset()
mydataset=DataLoader(dataset, batch_size=32, shuffle=True,num_workers=4)

In [ ]:

class forwardDiffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.timesteps = timesteps
        self.device = device

    
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1. - self.betas
        self.alpha_hat = torch.cumprod(self.alphas, dim=0) 

    def sample_timesteps(self, batch_size):
        return torch.randint(0, self.timesteps, (batch_size,)).to(self.device)

    def noise_images(self, x, t):
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]

        epsilon = torch.randn_like(x)

        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, stride=1):
        super().__init__()

        
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(32, out_ch)

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(32, out_ch)

        self.act = nn.SiLU()

    
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch)
        )

        # skip connection
        self.skip = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)

    def forward(self, x, t):

        identity = self.skip(x)

        # --------- time embedding injection ---------
        t_emb = self.time_mlp(t)              # (B, out_ch)
        t_emb = t_emb[:, :, None, None]       # (B, out_ch, 1, 1)

        # --------- first block ---------
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.act(h)

        # inject time here (after first conv like DDPM style)
        h = h + t_emb

        # --------- second block ---------
        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)

        # residual connection
        return h + identity

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        time=time.float()
        half_dim = self.dim // 2
        emb = torch.exp(
            torch.arange(half_dim, device=device) * -(torch.log(torch.tensor(10000.0)) / (half_dim - 1))
        )
        emb = time[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


In [ ]:
class DiffusionUNet(nn.Module):

    def __init__(self, time_dim=256):
        super().__init__()

        self.time_dim = time_dim

        self.time_embedding = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU()
)

        # ---------------- Encoder ----------------
        self.enc1 = ResBlock(3, 64, time_dim)
        self.down1 = nn.Conv2d(64, 64, 4, 2, 1)

        self.enc2 = ResBlock(64, 128, time_dim)
        self.down2 = nn.Conv2d(128, 128, 4, 2, 1)

        self.enc3 = ResBlock(128, 256, time_dim)

        # ---------------- Bottleneck ----------------
        self.mid = ResBlock(256, 256, time_dim)

        # ---------------- Decoder ----------------
        self.up1 = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.dec1 = ResBlock(256, 128, time_dim)  # concat -> 128+128

        self.up2 = nn.ConvTranspose2d(128, 64, 4, 2, 1)
        self.dec2 = ResBlock(128, 64, time_dim)   # concat -> 64+64

        self.out = nn.Conv2d(64, 3, 1)

    def forward(self, x, t):

        t = self.time_embedding(t)

        # ---------------- Encoder ----------------
        x1 = self.enc1(x, t)
        x2 = self.down1(x1)

        x3 = self.enc2(x2, t)
        x4 = self.down2(x3)

        x5 = self.enc3(x4, t)

        x_mid = self.mid(x5, t)

        # ------------------------- Decoder ----------------
        x = self.up1(x_mid)
        x = torch.cat([x, x3], dim=1)
        x = self.dec1(x, t)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec2(x, t)

        return self.out(x)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


forward_diffusion = forwardDiffusion(device=device)
diffusion=DiffusionUNet()


#forward_diffusion=torch.nn.DataParallel(forward_diffusion).to(device)
diffusion=torch.nn.DataParallel(diffusion).to(device)

scaler = torch.cuda.amp.GradScaler()



In [ ]:
learning_rate=0.0001
epochs=50

optimizer=Adam(diffusion.parameters(), lr=learning_rate)
loss=nn.MSELoss()

losses = []

In [ ]:
print("Starting Training Loop")

for epoch in range(epochs):
    diffusion.train()
    print(f"Epoch {epoch+1}/{epochs}")
    i = 0
    for batch in mydataset:

        batch=batch.to(device)

        optimizer.zero_grad()

        timestamp=forward_diffusion.sample_timesteps(batch_size=batch.shape[0])
        noisy_img,noise=forward_diffusion.noise_images(batch, timestamp)
            
        with torch.cuda.amp.autocast():

            noise_pred=diffusion(noisy_img, timestamp)
            loss_value=loss(noise_pred, noise)
        


        scaler.scale(loss_value).backward()
        scaler.step(optimizer)
        scaler.update()
        
        
        losses.append(loss_value.item())
        i+=1
        if i%100==0:
            print(f"Batch {i}, Loss: {loss_value.item():.4f}")
        

In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title("Diffusion Model Training Loss")
plt.xlabel("Training Iterations")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

In [ ]:
torch.save(diffusion.state_dict(), "diffusion_model.pth")